# Explicit list → Dask workers (minimal)

Illustrates the idea only — **no project helpers**. Needs:

- env: `S3_ENDPOINT`, `S3_BUCKET`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`
- optional: `OTEL_PREFIX` (default `otel-notebook`), `SPANS_DATE`, `DASK_SCHEDULER_ADDRESS`
- data already under `s3://$BUCKET/$PREFIX/spans/date=…/*.parquet`

Pattern: **list keys once → `dd.read_parquet(uris)` → `.compute()` on the Client**.



In [ ]:
# minimal: explicit LIST + distributed parquet read (no project imports)
import os, time
import s3fs
import dask
import dask.dataframe as dd
from dask.distributed import Client

# --- env (remote / JupyterHub / shell) ---
endpoint = os.environ["S3_ENDPOINT"]          # e.g. http://rustfs:9010
bucket   = os.environ["S3_BUCKET"]
prefix   = os.environ.get("OTEL_PREFIX", "otel-notebook").strip("/")
date     = (os.environ.get("SPANS_DATE") or "").strip() or None  # e.g. 2026-07-30
sched    = (
    os.environ.get("DASK_SCHEDULER_ADDRESS")
    or os.environ.get("DASK_SCHEDULER")  # only if it is tcp://… address
)
assert sched and "://" in sched, "set DASK_SCHEDULER_ADDRESS=tcp://scheduler:8786"

# DASK_SCHEDULER env must NOT be a tcp:// address (dask treats it as scheduler *type*)
if os.environ.get("DASK_SCHEDULER", "").startswith(("tcp://", "tls://")):
    os.environ.pop("DASK_SCHEDULER", None)
    dask.config.set({"scheduler": None})

storage_options = {
    "key": os.environ.get("AWS_ACCESS_KEY_ID"),
    "secret": os.environ.get("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs": {"endpoint_url": endpoint, "region_name": os.environ.get("AWS_REGION", "us-east-1")},
    "config_kwargs": {"s3": {"addressing_style": "path"}},  # path-style for MinIO/RustFS
}

# 1) list once (prune to one day when SPANS_DATE set)
root = f"{bucket}/{prefix}/spans"
if date:
    root = f"{root}/date={date}"
fs = s3fs.S3FileSystem(**storage_options)
keys = [k for k in fs.find(root) if k.endswith(".parquet")]
print(f"listed {len(keys)} files under s3://{root}/  sample={keys[:2]}")
assert keys, "no parquet — generate data or fix path/creds"

# 2) explicit URI list (not a bare glob on a huge tree)
uris = [f"s3://{k}" for k in keys]
with dask.config.set({"scheduler": "synchronous"}):  # plan on client
    ddf = dd.read_parquet(uris, storage_options=storage_options, aggregate_files=True)
    nparts = ddf.npartitions
print(f"partitions={nparts}  cols={list(ddf.columns)[:6]}")

# 3) workers do the read
client = Client(sched)
n_workers = len(client.scheduler_info().get("workers", {}))
print(f"workers={n_workers}  scheduler={sched}")
t0 = time.time()
nrows = int(ddf.shape[0].compute())  # not map_partitions(len).sum()
print(f"rows≈{nrows:,}  in {time.time()-t0:.1f}s on {n_workers} workers")
print("ok — explicit list + worker parquet read")

